<!-- NB_DOC_INTRO_V1 -->
# ML — Interpretabilite & Feature Selection (SHAP, LIME, ELI5, RFE, Boruta)

> 📚 **Doc thematique** : [docs/03_ML.md](docs/03_ML.md) (Machine Learning classique)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Comprendre pourquoi un modele predit ce qu'il predit** = clef du deploiement (audit reglementaire, debug, trust). Ce notebook execute les principales methodes d'**interpretabilite** (SHAP, LIME, permutation importance) et de **feature selection** (RFECV, Boruta, SelectFromModel).

Couvre aussi les **displays sklearn** modernes (ConfusionMatrix, ROC, Calibration, PartialDependence).

## Plan

1. Setup + modele de base
2. Feature importance built-in (impurity vs permutation)
3. SHAP (TreeExplainer + summary plot)
4. LIME (explication locale)
5. ELI5 (alternative)
6. Feature selection : SelectFromModel, RFECV
7. Boruta
8. Displays sklearn (ConfusionMatrix, ROC, Calibration, PartialDependence)
9. Pieges et anti-patterns
10. References


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import SelectFromModel, RFECV
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.calibration import CalibrationDisplay   # moved in sklearn 1.8
from sklearn.inspection import PartialDependenceDisplay
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
model.fit(X_tr, y_tr)
print(f"Test accuracy : {model.score(X_te, y_te):.4f}")
print(f"Nb features : {X.shape[1]}")

## 1. Feature importance built-in (impurity-based)

**Attention** : biaise vers les features a haute cardinalite. Toujours croiser avec permutation.

In [ ]:
imps = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
imps.head(15).plot(kind="barh", ax=ax)
ax.invert_yaxis()
ax.set_title("Top 15 features (impurity-based)")
plt.tight_layout(); plt.show()

## 2. Permutation importance — recommande, model-agnostic

In [ ]:
result = permutation_importance(model, X_te, y_te, n_repeats=10, random_state=SEED, n_jobs=-1)
imp_perm = pd.DataFrame({
    "feature": X.columns,
    "mean_drop_acc": result.importances_mean,
    "std": result.importances_std,
}).sort_values("mean_drop_acc", ascending=False)
print(imp_perm.head(10))

# Comparer impurity vs permutation (top 10)
top = imp_perm.head(10)["feature"].values
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(top))
w = 0.35
ax.barh(x - w/2, imps.reindex(top).values, w, label="Impurity")
ax.barh(x + w/2, imp_perm.set_index("feature").reindex(top)["mean_drop_acc"].values, w, label="Permutation")
ax.set_yticks(x); ax.set_yticklabels(top); ax.invert_yaxis()
ax.legend(); ax.set_title("Impurity vs Permutation importance")
plt.tight_layout(); plt.show()

## 3. SHAP — TreeExplainer + summary plot

In [ ]:
try:
    import shap
    print(f"SHAP : {shap.__version__}")

    # TreeExplainer : exact + tres rapide pour les arbres
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_te)

    # Shape : pour binaire on prend index 1 (classe positive)
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values
        if shap_vals.ndim == 3:
            shap_vals = shap_vals[:, :, 1]

    print(f"SHAP values shape : {shap_vals.shape}")

    # Summary plot (beeswarm)
    shap.summary_plot(shap_vals, X_te, show=False, max_display=10)
    plt.tight_layout(); plt.show()
except ImportError:
    print("SHAP pas installe (uv add --group ml shap)")

### Force plot — explication d'une prediction individuelle

In [ ]:
try:
    import shap
    # Pour 1 instance
    idx = 0
    print(f"Prediction pour patient {idx} : {model.predict(X_te.iloc[[idx]])[0]} "
          f"(proba {model.predict_proba(X_te.iloc[[idx]])[0, 1]:.3f})")

    # Bar plot des SHAP values pour cette instance
    instance_shap = shap_vals[idx]
    df_inst = pd.DataFrame({
        "feature": X.columns,
        "value": X_te.iloc[idx].values,
        "shap": instance_shap,
    }).sort_values("shap", key=abs, ascending=False).head(10)

    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["green" if v > 0 else "red" for v in df_inst["shap"]]
    ax.barh(df_inst["feature"], df_inst["shap"], color=colors)
    ax.invert_yaxis()
    ax.set_title(f"Explication SHAP — patient {idx}\n(positif = pousse vers benign, negatif = malign)")
    plt.tight_layout(); plt.show()
except (ImportError, NameError):
    print("SHAP skip")

## 4. LIME — explication locale

In [ ]:
try:
    from lime.lime_tabular import LimeTabularExplainer

    lime_exp = LimeTabularExplainer(
        training_data=X_tr.values,
        feature_names=X.columns.tolist(),
        class_names=["malign", "benign"],
        mode="classification",
        random_state=SEED,
    )

    # Expliquer 1 prediction
    idx = 0
    exp = lime_exp.explain_instance(X_te.iloc[idx].values, model.predict_proba, num_features=10)
    print(f"LIME — Top features (instance {idx}) :")
    for feat, weight in exp.as_list():
        print(f"  {feat:50s} : {weight:+.4f}")
except ImportError:
    print("LIME pas installe (uv add --group ml lime)")

## 5. Feature selection — `SelectFromModel`

In [ ]:
# SelectFromModel : garde les features avec importance > seuil (median par defaut)
selector = SelectFromModel(model, prefit=True, threshold="median")
X_tr_sel = selector.transform(X_tr)
print(f"Features retenues : {X_tr_sel.shape[1]} / {X.shape[1]}")
selected_features = X.columns[selector.get_support()].tolist()
print(f"  : {selected_features[:10]}")

# Re-entrainer sur features selectionnees
model_sel = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
model_sel.fit(X_tr_sel, y_tr)
print(f"\nAccuracy avec selection : {model_sel.score(selector.transform(X_te), y_te):.4f}")
print(f"Accuracy sans selection : {model.score(X_te, y_te):.4f}")

## 6. RFECV — Recursive Feature Elimination + CV

In [ ]:
selector = RFECV(
    estimator=RandomForestClassifier(n_estimators=50, random_state=SEED, n_jobs=-1),
    step=1, min_features_to_select=5, cv=3, scoring="roc_auc", n_jobs=-1,
)
selector.fit(X_tr, y_tr)
print(f"Nb optimal features : {selector.n_features_}")
print(f"Score CV avec : {max(selector.cv_results_['mean_test_score']):.4f}")
print(f"Features retenues : {X.columns[selector.support_].tolist()[:10]}")

## 7. Boruta — feature selection par comparaison aux 'shadow features'

In [ ]:
try:
    from boruta import BorutaPy

    boruta = BorutaPy(
        estimator=RandomForestClassifier(n_jobs=-1, random_state=SEED),
        n_estimators="auto",
        max_iter=50,
        random_state=SEED,
    )
    # BorutaPy attend des numpy arrays
    boruta.fit(X_tr.values, y_tr.values)
    print(f"Boruta confirmed : {(boruta.support_).sum()} features")
    print(f"Boruta tentative : {(boruta.support_weak_).sum()} features")
    print(f"Selected : {X.columns[boruta.support_].tolist()}")
except ImportError:
    print("Boruta pas installe (uv add --group ml boruta) — skip")

## 8. Displays sklearn

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

ConfusionMatrixDisplay.from_estimator(model, X_te, y_te, ax=axes[0], normalize="true",
                                       display_labels=["malign", "benign"])
axes[0].set_title("Confusion Matrix")

RocCurveDisplay.from_estimator(model, X_te, y_te, ax=axes[1])
axes[1].set_title("ROC Curve")

PrecisionRecallDisplay.from_estimator(model, X_te, y_te, ax=axes[2])
axes[2].set_title("PR Curve")

CalibrationDisplay.from_estimator(model, X_te, y_te, n_bins=10, ax=axes[3])
axes[3].set_title("Calibration")

plt.tight_layout(); plt.show()

### PartialDependenceDisplay — effet d'une feature, moyenne sur les autres

In [ ]:
top_features = imp_perm.head(4)["feature"].tolist()

PartialDependenceDisplay.from_estimator(model, X_te, top_features, kind="average")
plt.suptitle("Partial Dependence — top 4 features (impact sur P(benign))")
plt.tight_layout(); plt.show()

## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Conclure causalite a partir de SHAP | SHAP = correlationnel uniquement |
| Permutation importance sur train | Toujours sur **test** (sinon biaise) |
| Comparer SHAP entre 2 modeles differents | Pas calibres au meme baseline |
| Boruta sans `max_iter` | Peut tourner indefiniment |
| Selection feature SANS Pipeline en CV | Leakage : selection vue le test |
| 1 seule methode d'interpretation | Toujours croiser SHAP + permutation + Partial Dependence |
| SHAP TreeExplainer sur tres gros dataset | `shap.sample(X, 100)` pour sub-sample |


## References

### Documentation
- SHAP : https://shap.readthedocs.io/
- LIME : https://github.com/marcotcr/lime
- ELI5 : https://eli5.readthedocs.io/
- Boruta : https://github.com/scikit-learn-contrib/boruta_py
- sklearn inspection : https://scikit-learn.org/stable/modules/inspection.html
- *Interpretable Machine Learning* (Christoph Molnar) : https://christophm.github.io/interpretable-ml-book/

### Voir aussi
- [ML_Regression_Classification_CV_GridSearch.ipynb](ML_Regression_Classification_CV_GridSearch.ipynb)
- [ML_AutoML.ipynb](ML_AutoML.ipynb)
- [DS_Causal_Inference.ipynb](DS_Causal_Inference.ipynb)
- [DS_Metrics_Reference.ipynb](DS_Metrics_Reference.ipynb)


<!-- ENRICH_2025_V1 -->

## 🔥 Outils d'interpretabilite 2024-2025

### 1. **InterpretML — EBM** (Microsoft)

EBM = **Explainable Boosting Machine** : modele intrinsequement interpretable, performance proche du GBM, **chaque feature visualisable** comme une fonction du feature seul.

```python
# pip install interpret
from interpret.glassbox import ExplainableBoostingClassifier
ebm = ExplainableBoostingClassifier(random_state=42).fit(X_tr, y_tr)
from interpret import show
show(ebm.explain_global())   # dashboard interactif
```

### 2. **`dalex` — DrWhy.AI**

Alternative francais a SHAP/LIME, avec **diagnostic profond** : variable importance, **partial dependence**, **ALE** (Accumulated Local Effects), **Ceteris Paribus**.

```python
# pip install dalex
import dalex as dx
exp = dx.Explainer(model, X_tr, y_tr)
exp.model_performance().plot()
exp.model_parts().plot()           # variable importance
exp.predict_parts(X_te.iloc[0]).plot()   # break-down pour 1 prediction
```

### 3. **Captum** (PyTorch native)

Pour les DL : Integrated Gradients, DeepLift, GradCAM, LayerLRP.

```python
# pip install captum
from captum.attr import IntegratedGradients
ig = IntegratedGradients(model)
attributions = ig.attribute(input_tensor, target=class_idx)
```

### 4. **SHAP partition explainer** (2024+)

Pour les arbres : exact + 100× plus rapide que TreeExplainer classique sur gros datasets.

```python
import shap
explainer = shap.TreeExplainer(model)   # detecte automatiquement la meilleure strategie
# Si feature interactions importantes :
shap_interaction = explainer.shap_interaction_values(X[:100])
```

### 5. **Fairness — `fairlearn` MetricFrame**
```python
# pip install fairlearn
from fairlearn.metrics import MetricFrame, equalized_odds_difference
mf = MetricFrame(metrics={"acc": accuracy_score, "fpr": false_positive_rate},
                  y_true=y_te, y_pred=pred, sensitive_features=X_te["gender"])
print(mf.by_group)
print(f"Equalized odds diff : {equalized_odds_difference(y_te, pred, sensitive_features=X_te['gender']):.4f}")
```

### Comparatif rapide

| Outil | Forces | Limites |
|---|---|---|
| **SHAP** | Universel, theorie solide (Shapley), tres bonne UX | Lent sur gros datasets, attention multi-collinearite |
| **LIME** | Local, simple | Instable (random sampling), parfois trompeur |
| **InterpretML EBM** | Modele directement interpretable | Pas un wrapper, doit etre le modele |
| **dalex** | Diagnostic riche (ALE, Ceteris) | API francaise, moins repandu |
| **Captum** | DL native, GradCAM pour vision | Que PyTorch |
| **Eli5** | Permutation + viz tableau HTML | Plus limite que SHAP |
| **Boruta** | Selection robuste (shadow features) | Lent, parfois trop selectif |

> 🎯 **2025** : SHAP reste le standard, **InterpretML** monte vite pour les cas regulatoires (banque, sante) car interpretable by design.
